[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/19.%20The%20SLAP%20for%20Reefer%20Containers%20Problem/P19-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 19. The SLAP for Reefer Containers Problem
## Tier 2: First-Fit Decreasing Heuristic

### Purpose and inputs

The reefer-container storage location assignment problem (SLAP) asks where to place refrigerated containers while respecting the locations' power capacities and temperature ranges. This notebook applies a **First-Fit Decreasing (FFD)** heuristic to a small example. It sorts containers by power demand, from largest to smallest, and assigns each container to the first location that can accommodate it.

The input consists of containers with power demand, required temperature, dwell time, and cargo type, together with locations that have power capacity, an allowed temperature interval, and a base cost. The output is a greedy assignment, a record of the decisions, location utilization, and the resulting assignment cost. A container that cannot be placed by this procedure is reported as unassigned.

### Model assumptions and scope

- A container's required temperature must lie within the selected location's inclusive temperature interval.
- A location's power capacity limits the **sum** of the power demands of all containers assigned there. The model treats those demands as concurrent; it does not schedule arrival, departure, or changing power use over time.
- Dwell time affects assignment cost but does not affect power capacity or temperature compatibility.
- Each location has a fixed base-cost multiplier. The cost formula below is a modeling score in cost units; no currency or real-world tariff is specified.
- Containers are processed in descending order of power demand. Locations are considered in the order supplied. The procedure makes no reassignment and does not backtrack.
- Cargo type is descriptive only. Compatibility is determined by the numerical temperature requirement, not by a separate cargo-type rule.

This is a greedy feasibility procedure, not an optimization algorithm with an optimality guarantee. Its total cost is the sum of costs for assigned containers; unassigned containers receive no penalty in that total. Therefore, the reported cost alone does not measure how many containers were left unassigned.

### Mathematical model and FFD decision rule

Let $C$ be the set of containers and $L$ the set of locations. For container $i \in C$, let $e_i$ be its power demand, $t_i$ its required temperature, and $d_i$ its dwell time. For location $j \in L$, let $P_j$ be its power capacity, $[T_j^{\min}, T_j^{\max}]$ its allowed temperature interval, and $b_j$ its base-cost multiplier. A feasible assignment must obey

$$
\sum_{i \in C: x_{ij}=1} e_i \leq P_j \quad (j \in L),
\qquad
T_j^{\min} \leq t_i \leq T_j^{\max}
\quad \text{whenever } x_{ij}=1,
$$

where $x_{ij}=1$ means container $i$ is assigned to location $j$, and $x_{ij}=0$ otherwise. Each container is assigned to at most one location. FFD considers containers in non-increasing order of $e_i$. For each container, it scans locations in their supplied order and selects the first location satisfying both constraints. If no such location exists at that point in the scan, the container remains unassigned. This decision rule does not compare assignment costs when selecting a location.

The cost assigned to a container-location pair is

$$
c_{ij} = \frac{b_j e_i d_i}{1000},
\qquad
C_{\mathrm{assigned}} = \sum_{i \text{ assigned}} c_{i,\,a(i)},
$$

where $a(i)$ is the location chosen for container $i$. The factor $1000$ is a scaling factor in the specified cost formula. The algorithm reports this sum; it does not search for the assignment that minimizes it.

For $n=|C|$ containers and $m=|L|$ locations, sorting takes $O(n\log n)$ time and the location scan takes at most $O(nm)$ time. The implementation therefore takes $O(n\log n+nm)$ time and uses $O(n+m)$ additional storage, including its assignment records and location states. This describes computational effort, not solution quality.

### Example data

The example has four containers and three locations. The code below supplies their power demands, required temperatures, dwell times, location capacities, temperature intervals, and base-cost multipliers. The algorithm sorts the containers by power demand before making any assignments.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple
import time

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import numpy as np
import seaborn as sns

print("Packages imported successfully for the FFD heuristic.")

In [ ]:
@dataclass
class ReeferContainer:
    """Data describing one refrigerated container."""

    id: int
    energy: float  # Power demand in kW
    temp_req: float  # Required temperature in degrees Celsius
    dwell_time: float  # Dwell time in hours
    cargo_type: str  # Descriptive cargo category


@dataclass
class StorageLocation:
    """Capacity, temperature limits, and cost multiplier for a location."""

    id: int
    power_cap: float  # Power capacity in kW
    temp_min: float  # Minimum allowed temperature in degrees Celsius
    temp_max: float  # Maximum allowed temperature in degrees Celsius
    base_cost: float  # Multiplier used in the assignment-cost formula


@dataclass
class LocationState:
    """Mutable resource and cost totals maintained during assignment."""

    remaining_power: float
    temp_range: Tuple[float, float]
    assigned_containers: List[int]
    base_cost: float
    total_cost: float = 0.0


print("Data classes are ready for the assignment procedure.")

In [ ]:
containers = [
    ReeferContainer(
        id=1,
        energy=80,
        temp_req=-15,
        dwell_time=48,
        cargo_type="Pharmaceutical",
    ),
    ReeferContainer(
        id=2,
        energy=65,
        temp_req=3,
        dwell_time=72,
        cargo_type="Fresh Produce",
    ),
    ReeferContainer(
        id=3,
        energy=45,
        temp_req=-18,
        dwell_time=96,
        cargo_type="Frozen Seafood",
    ),
    ReeferContainer(
        id=4,
        energy=70,
        temp_req=5,
        dwell_time=60,
        cargo_type="Dairy Products",
    ),
]

locations = [
    StorageLocation(
        id=1, power_cap=100, temp_min=-25, temp_max=8, base_cost=12
    ),
    StorageLocation(
        id=2, power_cap=85, temp_min=-20, temp_max=10, base_cost=10
    ),
    StorageLocation(
        id=3, power_cap=120, temp_min=-30, temp_max=-5, base_cost=15
    ),
]

print(f"Example instance: {len(containers)} containers and {len(locations)} locations")
print("\nContainers in FFD processing order (descending power demand):")
for container in sorted(containers, key=lambda item: item.energy, reverse=True):
    print(
        f"  C{container.id}: {container.cargo_type}, "
        f"{container.energy} kW, {container.temp_req} °C, "
        f"{container.dwell_time} h"
    )

print("\nStorage locations in scan order:")
for location in locations:
    print(
        f"  L{location.id}: {location.power_cap} kW, "
        f"[{location.temp_min} °C, {location.temp_max} °C], "
        f"base-cost multiplier={location.base_cost}"
    )

In [ ]:
def check_temperature_compatibility(
    container: ReeferContainer,
    location: StorageLocation,
) -> bool:
    """Return whether the required temperature lies in the location interval."""
    return location.temp_min <= container.temp_req <= location.temp_max


def calculate_assignment_cost(
    container: ReeferContainer,
    location: StorageLocation,
) -> float:
    """Apply the specified base-cost, power-demand, and dwell-time formula."""
    return location.base_cost * container.energy * container.dwell_time / 1000


def assign_reefer_containers_ffd(
    containers: List[ReeferContainer],
    locations: List[StorageLocation],
) -> Dict:
    """Assign containers greedily using First-Fit Decreasing.

    Containers are sorted by descending power demand. Each is assigned to
    the first location, in the supplied location order, with sufficient
    remaining power and a compatible temperature interval.
    """
    container_ids = [container.id for container in containers]
    location_ids = [location.id for location in locations]
    if len(container_ids) != len(set(container_ids)):
        raise ValueError("Container IDs must be unique.")
    if len(location_ids) != len(set(location_ids)):
        raise ValueError("Location IDs must be unique.")

    # Stable sorting preserves input order when containers have equal demand.
    sorted_containers = sorted(
        containers,
        key=lambda container: container.energy,
        reverse=True,
    )

    # The ID map keeps the location scan in the supplied order without
    # repeatedly searching the location list inside the container loop.
    locations_by_id = {location.id: location for location in locations}
    location_states = {
        location.id: LocationState(
            remaining_power=location.power_cap,
            temp_range=(location.temp_min, location.temp_max),
            assigned_containers=[],
            base_cost=location.base_cost,
        )
        for location in locations
    }

    assignment = {}
    total_cost = 0.0
    assignment_log = []
    unassigned_containers = []

    for container in sorted_containers:
        assigned = False

        # First-fit means the scan stops at the first location satisfying
        # both resource capacity and temperature compatibility.
        for location_id, location_state in location_states.items():
            location = locations_by_id[location_id]
            power_before = location_state.remaining_power
            power_feasible = power_before >= container.energy
            temp_feasible = check_temperature_compatibility(
                container,
                location,
            )

            if power_feasible and temp_feasible:
                cost = calculate_assignment_cost(container, location)
                assignment[container.id] = location_id
                location_state.remaining_power -= container.energy
                location_state.assigned_containers.append(container.id)
                location_state.total_cost += cost
                total_cost += cost

                assignment_log.append(
                    {
                        "container_id": container.id,
                        "container_energy": container.energy,
                        "container_temp": container.temp_req,
                        "assigned_location": location_id,
                        "location_remaining_before": power_before,
                        "location_remaining_after": location_state.remaining_power,
                        "assignment_cost": cost,
                        "decision": (
                            f"Assigned C{container.id} to L{location_id} "
                            f"(power feasible: {power_feasible}; "
                            f"temperature feasible: {temp_feasible})"
                        ),
                    }
                )
                assigned = True
                break

        if not assigned:
            unassigned_containers.append(container.id)
            assignment_log.append(
                {
                    "container_id": container.id,
                    "container_energy": container.energy,
                    "container_temp": container.temp_req,
                    "assigned_location": None,
                    "decision": (
                        f"C{container.id} could not be assigned: "
                        "no feasible location was found during its scan"
                    ),
                }
            )

    return {
        "assignment": assignment,
        "total_cost": total_cost,
        "location_states": location_states,
        "assignment_log": assignment_log,
        "unassigned_containers": unassigned_containers,
        "algorithm": "First-Fit Decreasing",
    }


print("The FFD assignment function is ready.")

### Run the greedy assignment

The next cell runs the procedure on the example data. The order of the locations in `locations` matters: FFD selects the first feasible location, not necessarily the cheapest one. The displayed execution time is a measurement from the current run and can vary with the computing environment.

In [ ]:
start_time = time.perf_counter()
ffd_result = assign_reefer_containers_ffd(containers, locations)
execution_time = time.perf_counter() - start_time

print("=== FIRST-FIT DECREASING RESULTS ===")
print(f"Algorithm: {ffd_result['algorithm']}")
print(f"Execution time: {execution_time:.6f} seconds")
print(f"Assigned cost units: {ffd_result['total_cost']:.2f}")
print(f"Containers assigned: {len(ffd_result['assignment'])}/{len(containers)}")
print(f"Unassigned container IDs: {ffd_result['unassigned_containers']}")

containers_by_id = {container.id: container for container in containers}
locations_by_id = {location.id: location for location in locations}

print("\nAssignments:")
for container_id, location_id in ffd_result["assignment"].items():
    container = containers_by_id[container_id]
    location = locations_by_id[location_id]
    print(
        f"  C{container_id} ({container.cargo_type}) -> L{location_id}"
    )
    print(
        f"    Power demand: {container.energy} kW; "
        f"required temperature: {container.temp_req} °C"
    )
    print(
        f"    Location temperature range: "
        f"[{location.temp_min} °C, {location.temp_max} °C]; "
        f"power capacity: {location.power_cap} kW"
    )

print("\nLocation utilization:")
for location in locations:
    state = ffd_result["location_states"][location.id]
    power_used = location.power_cap - state.remaining_power
    utilization = power_used / location.power_cap * 100
    print(
        f"  L{location.id}: {power_used:.1f}/{location.power_cap:.1f} kW "
        f"({utilization:.1f}%)"
    )
    print(f"    Assigned containers: {state.assigned_containers}")
    print(f"    Cost contribution: {state.total_cost:.2f} cost units")

### Inspect decisions and verify feasibility

The decision log shows the processing order and the remaining capacity after each successful placement. The checks below verify that the returned assignment respects the model's constraints and that the reported total equals the sum of the location-level cost contributions. An unassigned container is not automatically evidence that the instance has no feasible complete assignment: the greedy choices made earlier may affect what remains available.

In [ ]:
import math

print("=== ASSIGNMENT DECISION LOG ===")
print("Container processing order: descending power demand")
for step, log_entry in enumerate(ffd_result["assignment_log"], start=1):
    print(f"{step}. {log_entry['decision']}")
    if log_entry["assigned_location"] is not None:
        print(
            f"   Assignment cost: {log_entry['assignment_cost']:.2f} cost units; "
            f"remaining power: "
            f"{log_entry['location_remaining_after']:.1f} kW"
        )

assigned_container_ids = set(ffd_result["assignment"])
unassigned_container_ids = set(ffd_result["unassigned_containers"])
all_container_ids = {container.id for container in containers}
assert assigned_container_ids.isdisjoint(unassigned_container_ids)
assert assigned_container_ids | unassigned_container_ids == all_container_ids

capacity_violations = []
for location in locations:
    state = ffd_result["location_states"][location.id]
    power_used = location.power_cap - state.remaining_power
    if power_used > location.power_cap and not math.isclose(
        power_used,
        location.power_cap,
        rel_tol=1e-9,
        abs_tol=1e-9,
    ):
        capacity_violations.append(location.id)

temperature_violations = []
for container_id, location_id in ffd_result["assignment"].items():
    container = containers_by_id[container_id]
    location = locations_by_id[location_id]
    if not check_temperature_compatibility(container, location):
        temperature_violations.append(container_id)

cost_from_locations = sum(
    state.total_cost
    for state in ffd_result["location_states"].values()
)
assert math.isclose(
    ffd_result["total_cost"],
    cost_from_locations,
    rel_tol=1e-9,
    abs_tol=1e-9,
)

print("\n=== FEASIBILITY AND SUMMARY ===")
if capacity_violations:
    print(f"Power-capacity violations at location IDs: {capacity_violations}")
else:
    print("All assigned power demands respect location capacities.")

if temperature_violations:
    print(f"Temperature violations for container IDs: {temperature_violations}")
else:
    print("All assigned containers meet their location temperature limits.")

assignment_rate = len(ffd_result["assignment"]) / len(containers) * 100
average_location_utilization = np.mean(
    [
        (location.power_cap - ffd_result["location_states"][location.id].remaining_power)
        / location.power_cap
        * 100
        for location in locations
    ]
)
assigned_count = len(ffd_result["assignment"])
cost_per_assigned_container = (
    ffd_result["total_cost"] / assigned_count if assigned_count else 0.0
)

print(f"Assignment rate: {assignment_rate:.1f}%")
print(
    "Average utilization across all locations: "
    f"{average_location_utilization:.1f}%"
)
print(
    "Cost per assigned container: "
    f"{cost_per_assigned_container:.2f} cost units"
)

### Visualize assignments and resource use

The four panels show the FFD processing order, location power use, how the assigned cost is distributed, and which temperature intervals are compatible. A red outline and check mark identify an actual assignment in the compatibility matrix. The heatmap's color key distinguishes temperature-compatible and incompatible pairs; it does not encode power feasibility.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle(
    "First-Fit Decreasing: Reefer Container Assignment",
    fontsize=16,
    fontweight="bold",
)

# Panel 1: power demand in the order containers are processed.
ax1 = axes[0, 0]
sorted_containers = sorted(
    containers,
    key=lambda container: container.energy,
    reverse=True,
)
assignment_order = [
    ffd_result["assignment"].get(container.id)
    for container in sorted_containers
]
bar_colors = [
    "seagreen" if location_id is not None else "firebrick"
    for location_id in assignment_order
]
x_positions = np.arange(len(sorted_containers))
bars = ax1.bar(
    x_positions,
    [container.energy for container in sorted_containers],
    color=bar_colors,
    alpha=0.8,
    edgecolor="black",
)
ax1.set_xlabel("Container processing order")
ax1.set_ylabel("Power demand (kW)")
ax1.set_title("FFD Order and Assignment")
ax1.set_xticks(x_positions)
ax1.set_xticklabels([f"C{container.id}" for container in sorted_containers])
ax1.grid(True, axis="y", alpha=0.3)
for bar, location_id in zip(bars, assignment_order):
    label = f"L{location_id}" if location_id is not None else "Unassigned"
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 2,
        label,
        ha="center",
        va="bottom",
        fontsize=9,
    )

# Panel 2: assigned power compared with each location's capacity.
ax2 = axes[0, 1]
location_names = [f"L{location.id}" for location in locations]
power_used = [
    location.power_cap - ffd_result["location_states"][location.id].remaining_power
    for location in locations
]
power_capacity = [location.power_cap for location in locations]
x = np.arange(len(locations))
width = 0.36
ax2.bar(
    x - width / 2,
    power_used,
    width,
    label="Power assigned",
    color="steelblue",
    alpha=0.85,
)
ax2.bar(
    x + width / 2,
    power_capacity,
    width,
    label="Power capacity",
    color="lightgray",
    edgecolor="black",
    alpha=0.85,
)
ax2.set_xlabel("Storage location")
ax2.set_ylabel("Power (kW)")
ax2.set_title("Assigned Power and Capacity")
ax2.set_xticks(x)
ax2.set_xticklabels(location_names)
ax2.legend()
ax2.grid(True, axis="y", alpha=0.3)
for index, (used, capacity) in enumerate(zip(power_used, power_capacity)):
    utilization = used / capacity * 100
    ax2.text(
        index - width / 2,
        used + 2,
        f"{utilization:.1f}%",
        ha="center",
        va="bottom",
        fontsize=9,
    )

# Panel 3: the cost contributions produced by the assignments.
ax3 = axes[1, 0]
location_costs = [
    ffd_result["location_states"][location.id].total_cost
    for location in locations
]
location_counts = [
    len(ffd_result["location_states"][location.id].assigned_containers)
    for location in locations
]
cost_labels = [
    f"L{location.id} ({count} containers)"
    for location, count in zip(locations, location_counts)
]
ax3.pie(
    location_costs,
    labels=cost_labels,
    autopct="%1.1f%%",
    startangle=90,
)
ax3.set_title("Share of Assigned Cost by Location")

# Panel 4: temperature compatibility, with actual assignments outlined.
ax4 = axes[1, 1]
compatibility_matrix = np.zeros((len(sorted_containers), len(locations)))
for row, container in enumerate(sorted_containers):
    for column, location in enumerate(locations):
        compatibility_matrix[row, column] = check_temperature_compatibility(
            container,
            location,
        )

heatmap = sns.heatmap(
    compatibility_matrix,
    annot=False,
    cmap=ListedColormap(["#f2f2f2", "#9bd3ae"]),
    cbar=False,
    linewidths=0.5,
    linecolor="white",
    xticklabels=[
        f"L{location.id}\n[{location.temp_min} °C, {location.temp_max} °C]"
        for location in locations
    ],
    yticklabels=[
        f"C{container.id}: {container.temp_req} °C"
        for container in sorted_containers
    ],
    ax=ax4,
)
for row, container in enumerate(sorted_containers):
    location_id = ffd_result["assignment"].get(container.id)
    if location_id is not None:
        column = next(
            index
            for index, location in enumerate(locations)
            if location.id == location_id
        )
        ax4.add_patch(
            plt.Rectangle(
                (column, row),
                1,
                1,
                fill=False,
                edgecolor="darkred",
                linewidth=2.5,
            )
        )
        ax4.text(
            column + 0.5,
            row + 0.5,
            "✓",
            ha="center",
            va="center",
            fontsize=14,
            color="darkred",
        )
ax4.set_title("Temperature Compatibility and Assignment")
ax4.set_xlabel("Storage location and allowed temperature range")
ax4.set_ylabel("Container and required temperature")
ax4.legend(
    handles=[
        Patch(facecolor="#9bd3ae", label="Temperature compatible"),
        Patch(facecolor="#f2f2f2", label="Temperature incompatible"),
        Patch(
            facecolor="white",
            edgecolor="darkred",
            linewidth=2,
            label="Assigned pair",
        ),
    ],
    loc="upper left",
    bbox_to_anchor=(0, -0.22),
    ncol=1,
    fontsize=8,
)

plt.tight_layout()
plt.show()

### Check cost accounting

This section makes the cost calculation transparent by listing each assigned container's contribution. The check connects the pairwise formula $c_{ij}=b_j e_i d_i/1000$ to the total returned by the assignment function. It is an accounting check, not a search for a lower-cost assignment.

In [ ]:
print("=== ASSIGNMENT COST ACCOUNTING ===")
recomputed_total_cost = 0.0

for container in sorted_containers:
    location_id = ffd_result["assignment"].get(container.id)
    if location_id is None:
        print(f"C{container.id}: unassigned; no assignment cost is included.")
        continue

    location = locations_by_id[location_id]
    cost = calculate_assignment_cost(container, location)
    recomputed_total_cost += cost
    print(
        f"C{container.id} -> L{location_id}: "
        f"({location.base_cost} x {container.energy} kW x "
        f"{container.dwell_time} h) / 1000 = {cost:.2f} cost units"
    )

assert math.isclose(
    recomputed_total_cost,
    ffd_result["total_cost"],
    rel_tol=1e-9,
    abs_tol=1e-9,
)
print(f"Recomputed assigned cost: {recomputed_total_cost:.2f} cost units")
print(f"Reported assigned cost:   {ffd_result['total_cost']:.2f} cost units")

### Scalability experiment

The following experiment generates reproducible synthetic instances and runs the same FFD function on each one. The seed is fixed inside the data-generation function so each requested instance size is generated reproducibly without changing NumPy's global random-number state. The generated instances illustrate runtime and assignment outcomes for this particular synthetic data process; they do not establish a general assignment-rate or cost guarantee.

In [ ]:
def generate_problem_instance(
    n_containers: int,
    n_locations: int,
    seed: int = 42,
) -> Tuple[List[ReeferContainer], List[StorageLocation]]:
    """Generate a reproducible synthetic instance for the FFD experiment."""
    rng = np.random.RandomState(seed)
    cargo_types = [
        "Pharmaceutical",
        "Fresh Produce",
        "Frozen Seafood",
        "Dairy",
        "Chemicals",
    ]
    generated_containers = []

    for index in range(n_containers):
        cargo_type = rng.choice(cargo_types)

        if cargo_type == "Pharmaceutical":
            energy = rng.uniform(30, 60)
            temp_req = rng.uniform(-25, -10)
        elif cargo_type == "Fresh Produce":
            energy = rng.uniform(25, 50)
            temp_req = rng.uniform(0, 8)
        elif cargo_type == "Frozen Seafood":
            energy = rng.uniform(40, 80)
            temp_req = rng.uniform(-25, -15)
        elif cargo_type == "Dairy":
            energy = rng.uniform(20, 45)
            temp_req = rng.uniform(0, 6)
        else:
            energy = rng.uniform(35, 70)
            temp_req = rng.uniform(-20, 5)

        generated_containers.append(
            ReeferContainer(
                id=index + 1,
                energy=round(energy, 1),
                temp_req=round(temp_req, 1),
                dwell_time=rng.uniform(24, 168),
                cargo_type=cargo_type,
            )
        )

    generated_locations = []
    for index in range(n_locations):
        if index % 3 == 0:
            temp_min, temp_max = -30, -5
            power_cap = rng.uniform(80, 150)
        elif index % 3 == 1:
            temp_min, temp_max = -10, 15
            power_cap = rng.uniform(60, 120)
        else:
            temp_min, temp_max = -20, 10
            power_cap = rng.uniform(70, 130)

        generated_locations.append(
            StorageLocation(
                id=index + 1,
                power_cap=round(power_cap, 1),
                temp_min=temp_min,
                temp_max=temp_max,
                base_cost=rng.uniform(8, 20),
            )
        )

    return generated_containers, generated_locations


test_sizes = [
    (5, 3),
    (10, 5),
    (20, 8),
    (50, 15),
    (100, 20),
]

scalability_results = []
for n_containers, n_locations in test_sizes:
    test_containers, test_locations = generate_problem_instance(
        n_containers,
        n_locations,
    )

    start_time = time.perf_counter()
    result = assign_reefer_containers_ffd(test_containers, test_locations)
    elapsed_time = time.perf_counter() - start_time

    assigned_count = len(result["assignment"])
    scalability_results.append(
        {
            "n_containers": n_containers,
            "n_locations": n_locations,
            "total_cost": result["total_cost"],
            "assigned_count": assigned_count,
            "assignment_rate": assigned_count / n_containers * 100,
            "execution_time": elapsed_time,
            "time_per_container_ms": elapsed_time / n_containers * 1000,
        }
    )

    print(
        f"{n_containers} containers, {n_locations} locations: "
        f"{assigned_count}/{n_containers} assigned; "
        f"cost={result['total_cost']:.2f} cost units; "
        f"runtime={elapsed_time:.6f} seconds"
    )

problem_labels = [
    f"{row['n_containers']}C, {row['n_locations']}L"
    for row in scalability_results
]
problem_positions = np.arange(len(scalability_results))

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle("FFD Scalability on Synthetic Instances", fontsize=16, fontweight="bold")

ax1 = axes[0, 0]
runtimes = [row["execution_time"] for row in scalability_results]
ax1.plot(problem_positions, runtimes, marker="o", color="navy", linewidth=2)
ax1.set_xlabel("Problem size (containers, locations)")
ax1.set_ylabel("Runtime (seconds)")
ax1.set_title("Measured Runtime")
ax1.set_xticks(problem_positions)
ax1.set_xticklabels(problem_labels, rotation=30, ha="right")
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
assignment_rates = [row["assignment_rate"] for row in scalability_results]
ax2.bar(problem_positions, assignment_rates, color="mediumseagreen", edgecolor="black")
ax2.set_xlabel("Problem size (containers, locations)")
ax2.set_ylabel("Assigned containers (%)")
ax2.set_title("Assignment Rate")
ax2.set_xticks(problem_positions)
ax2.set_xticklabels(problem_labels, rotation=30, ha="right")
ax2.set_ylim(0, 105)
ax2.grid(True, axis="y", alpha=0.3)

ax3 = axes[1, 0]
time_per_container = [
    row["time_per_container_ms"] for row in scalability_results
]
ax3.bar(problem_positions, time_per_container, color="salmon", edgecolor="black")
ax3.set_xlabel("Problem size (containers, locations)")
ax3.set_ylabel("Runtime per input container (ms)")
ax3.set_title("Runtime per Input Container")
ax3.set_xticks(problem_positions)
ax3.set_xticklabels(problem_labels, rotation=30, ha="right")
ax3.grid(True, axis="y", alpha=0.3)

ax4 = axes[1, 1]
assigned_counts = [row["assigned_count"] for row in scalability_results]
cost_per_assigned = [
    row["total_cost"] / row["assigned_count"]
    if row["assigned_count"]
    else 0.0
    for row in scalability_results
]
ax4.bar(problem_positions, cost_per_assigned, color="lightskyblue", edgecolor="black")
ax4.set_xlabel("Problem size (containers, locations)")
ax4.set_ylabel("Cost units per assigned container")
ax4.set_title("Mean Cost per Assigned Container")
ax4.set_xticks(problem_positions)
ax4.set_xticklabels(problem_labels, rotation=30, ha="right")
ax4.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## Interpretation and limitations

FFD is useful when a quick, transparent assignment is required and its greedy result is acceptable as a practical starting point. Sorting prioritizes containers with larger power demands, while the first-feasible-location rule makes each placement easy to trace. The resulting assignment depends on both the sorted container order and the supplied location order.

The method does not guarantee that it will assign every container even when another arrangement could do so, and it does not guarantee minimum assignment cost. It also does not backtrack, reassign containers, or model time-dependent arrivals, departures, power profiles, or temperature changes. These limitations follow from the stated model and decision rule; the utilization and cost reports should be interpreted within that scope.

The scalability experiment reports measured runtimes and outcomes for its generated instances. Runtime depends on the computing environment, and assignment rates and costs depend on the data. The complexity bound for the implemented procedure is $O(n\log n+nm)$, where $n$ is the number of containers and $m$ is the number of locations.